In [1]:
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor
import numpy as np
from joblib import dump, load
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parents[1]
COMBINED_DIR = BASE_DIR / "Data" / "combined"
MODEL_DIR = BASE_DIR / "src" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("COMBINED_DIR exists:", COMBINED_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())

BASE_DIR: /Users/m.mughees/Desktop/Pi-515-2026
COMBINED_DIR exists: True
MODEL_DIR exists: True


In [3]:
combined_train_df = pd.read_csv(COMBINED_DIR / "combined_train.csv")
smos_test_df = pd.read_csv(COMBINED_DIR / "smos_test_aligned.csv")
scan_holdout_df = pd.read_csv(COMBINED_DIR / "scan_holdout_aligned.csv")

print("Combined train:", combined_train_df.shape)
print("SMOS test:", smos_test_df.shape)
print("SCAN holdout:", scan_holdout_df.shape)

combined_train_df.head()

Combined train: (2103894, 11)
SMOS test: (10000, 11)
SCAN holdout: (452, 11)


,latitude,longitude,air_temperature,dewpoint_temperature,total_precipitation,year,month,day,soil_temperature,soil_moisture,source
0,-0.776986,1.426464,-1.215082,-0.989945,-0.367620,-0.314381,-1.292227,-1.323847,-1.369474,0.360234,SMOS
1,-1.216539,0.035936,0.823547,0.796616,-0.359515,0.372430,0.014939,1.733197,1.056937,0.213469,SMOS
2,-1.028159,1.205244,0.467160,0.505021,-0.336933,-1.230129,-0.965435,1.053854,0.334309,0.166913,SMOS
3,0.478879,-0.912151,-0.537826,-0.400612,-0.368519,-1.230129,1.322104,-1.663519,-0.674495,0.199984,SMOS
4,-0.400226,0.035936,0.926438,1.085233,-0.078794,-1.459067,0.014939,-0.191609,0.985257,0.214700,SMOS


In [5]:
def identity_function(X):
    return X

In [6]:
soil_temp_model = load(MODEL_DIR / "combined_soil_temperature_model.joblib")
print("✅ Loaded combined soil temperature model")

✅ Loaded combined soil temperature model


In [7]:
base_features = [
    "latitude",
    "longitude",
    "air_temperature",
    "dewpoint_temperature",
    "total_precipitation",
    "year",
    "month",
    "day",
]

target = "soil_moisture"

In [8]:
X_full = combined_train_df[base_features]
y_full = combined_train_df[target]

X_train, X_dev, y_train, y_dev = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42
)

X_smos_test = smos_test_df[base_features]
y_smos_test = smos_test_df[target]

X_scan_test = scan_holdout_df[base_features]
y_scan_test = scan_holdout_df[target]

print("✅ Data Split Shapes:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)
print("  X_smos_test:", X_smos_test.shape)
print("  X_scan_test:", X_scan_test.shape)
print("  y_train:", y_train.shape)
print("  y_dev:", y_dev.shape)
print("  y_smos_test:", y_smos_test.shape)
print("  y_scan_test:", y_scan_test.shape)

✅ Data Split Shapes:
  X_train: (1683115, 8)
  X_dev: (420779, 8)
  X_smos_test: (10000, 8)
  X_scan_test: (452, 8)
  y_train: (1683115,)
  y_dev: (420779,)
  y_smos_test: (10000,)
  y_scan_test: (452,)


In [9]:
train_pred_soil_temp = soil_temp_model.predict(X_train)
dev_pred_soil_temp = soil_temp_model.predict(X_dev)
smos_test_pred_soil_temp = soil_temp_model.predict(X_smos_test)
scan_test_pred_soil_temp = soil_temp_model.predict(X_scan_test)

print(train_pred_soil_temp[:5])

[ 1.14988     1.1761762   0.6127735   0.36312494 -0.82037807]


In [10]:
X_train = X_train.copy()
X_dev = X_dev.copy()
X_smos_test = X_smos_test.copy()
X_scan_test = X_scan_test.copy()

X_train["predicted_soil_temp"] = train_pred_soil_temp
X_dev["predicted_soil_temp"] = dev_pred_soil_temp
X_smos_test["predicted_soil_temp"] = smos_test_pred_soil_temp
X_scan_test["predicted_soil_temp"] = scan_test_pred_soil_temp

X_train.head()

,latitude,longitude,air_temperature,dewpoint_temperature,total_precipitation,year,month,day,predicted_soil_temp
232014,-1.028159,-0.785740,0.922408,1.191540,1.166028,1.288179,0.014939,-1.663519,1.149880
1806138,0.478879,-0.374902,1.059206,1.111577,-0.368666,0.830305,0.014939,0.600958,1.176176
1176116,0.039326,1.205244,0.596801,0.836222,-0.361066,-0.314381,0.014939,-1.323847,0.612773
1886990,-0.588606,0.889214,0.452993,0.126645,-0.368588,1.059242,-0.638644,0.034839,0.363125
601666,-1.028159,1.205244,-0.668904,-0.400420,-0.202768,-0.314381,1.322104,1.280301,-0.820378


In [11]:
def evaluate_xgb(X_train, y_train, X_dev, y_dev):
    print("Evaluating XGBoost Regressor for Combined Soil Moisture...")

    param_grid = {
        "n_estimators": [100, 300, 500, 1000],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
    }

    grid_search = GridSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=42),
        param_grid,
        cv=3,
        scoring="neg_mean_squared_error",
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    best_estimator = grid_search.best_estimator_

    try:
        importances = best_estimator.feature_importances_
        feature_df = pd.DataFrame({
            "Feature": X_train.columns,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)

        print("\nTop Features:")
        print(feature_df.head(10))

    except Exception as e:
        print("Could not extract feature importances:", e)

    y_pred = best_estimator.predict(X_dev)
    rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
    mae = mean_absolute_error(y_dev, y_pred)
    r2 = r2_score(y_dev, y_pred)

    print("\nGrid searching is done!")
    print("Best score (neg MSE):", grid_search.best_score_)
    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return best_estimator, rmse, mae, r2

In [12]:
def evaluate_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mean_target = np.mean(y_true)

    print(f"\n📊 {label} Set Performance:")
    print(f"Mean of y_{label.lower()}: {mean_target:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")

    return rmse, mae, r2

In [13]:
print("\n🚀 Evaluating model for: Combined Soil Moisture")

best_model, _, _, _ = evaluate_xgb(X_train, y_train, X_dev, y_dev)


🚀 Evaluating model for: Combined Soil Moisture
Evaluating XGBoost Regressor for Combined Soil Moisture...
Fitting 3 folds for each of 32 candidates, totalling 96 fits

Top Features:
                Feature  Importance
8   predicted_soil_temp    0.364736
5                  year    0.114729
0              latitude    0.103867
2       air_temperature    0.100333
6                 month    0.094110
4   total_precipitation    0.078098
3  dewpoint_temperature    0.068919
1             longitude    0.037744
7                   day    0.037465

Grid searching is done!
Best score (neg MSE): -0.0023515344495263837
Best hyperparameters:
{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 1000, 'subsample': 0.8}


In [14]:
y_train_pred = best_model.predict(X_train)
y_dev_pred = best_model.predict(X_dev)
y_smos_test_pred = best_model.predict(X_smos_test)
y_scan_test_pred = best_model.predict(X_scan_test)

In [15]:
train_rmse, train_mae, train_r2 = evaluate_metrics(y_train, y_train_pred, "Train")
dev_rmse, dev_mae, dev_r2 = evaluate_metrics(y_dev, y_dev_pred, "Dev")
smos_test_rmse, smos_test_mae, smos_test_r2 = evaluate_metrics(y_smos_test, y_smos_test_pred, "SMOS Test")
scan_test_rmse, scan_test_mae, scan_test_r2 = evaluate_metrics(y_scan_test, y_scan_test_pred, "SCAN Holdout")


📊 Train Set Performance:
Mean of y_train: 0.1799
RMSE: 0.0481
MAE: 0.0353
R²: 0.6865

📊 Dev Set Performance:
Mean of y_dev: 0.1799
RMSE: 0.0484
MAE: 0.0355
R²: 0.6818

📊 SMOS Test Set Performance:
Mean of y_smos test: 0.1770
RMSE: 0.0472
MAE: 0.0349
R²: 0.6880

📊 SCAN Holdout Set Performance:
Mean of y_scan holdout: 0.2924
RMSE: 0.0668
MAE: 0.0515
R²: 0.6605


In [16]:
results_smos_df = pd.DataFrame({
    "actual_soil_moisture": y_smos_test.values,
    "predicted_soil_moisture": y_smos_test_pred,
    "source": "SMOS"
})

results_scan_df = pd.DataFrame({
    "actual_soil_moisture": y_scan_test.values,
    "predicted_soil_moisture": y_scan_test_pred,
    "source": "SCAN"
})

pd.concat([results_smos_df.head(5), results_scan_df.head(5)], ignore_index=True)

,actual_soil_moisture,predicted_soil_moisture,source
0,0.178049,0.165967,SMOS
1,0.159784,0.165215,SMOS
2,0.189298,0.189737,SMOS
3,0.149692,0.153828,SMOS
4,0.141275,0.134670,SMOS
5,0.500917,0.441095,SCAN
6,0.120750,0.208024,SCAN
7,0.235625,0.281479,SCAN
8,0.437500,0.348124,SCAN
9,0.400417,0.316979,SCAN


In [17]:
dump(best_model, MODEL_DIR / "combined_soil_moisture_model.joblib")

print("✅ Model saved as:", MODEL_DIR / "combined_soil_moisture_model.joblib")

✅ Model saved as: /Users/m.mughees/Desktop/Pi-515-2026/src/models/combined_soil_moisture_model.joblib


In [18]:
metrics_df = pd.DataFrame([
    {"set": "train", "rmse": train_rmse, "mae": train_mae, "r2": train_r2},
    {"set": "dev", "rmse": dev_rmse, "mae": dev_mae, "r2": dev_r2},
    {"set": "smos_test", "rmse": smos_test_rmse, "mae": smos_test_mae, "r2": smos_test_r2},
    {"set": "scan_holdout", "rmse": scan_test_rmse, "mae": scan_test_mae, "r2": scan_test_r2},
])

all_results_df = pd.concat([results_smos_df, results_scan_df], ignore_index=True)

metrics_df.to_csv(MODEL_DIR / "combined_soil_moisture_metrics.csv", index=False)
all_results_df.to_csv(MODEL_DIR / "combined_soil_moisture_predictions.csv", index=False)

print("✅ Metrics and predictions saved.")

✅ Metrics and predictions saved.


In [19]:
metrics_df

,set,rmse,mae,r2
0,train,0.048083,0.035279,0.686511
1,dev,0.048440,0.035517,0.681844
2,smos_test,0.047188,0.034874,0.687996
3,scan_holdout,0.066797,0.051539,0.660532
